<table>
  <tr>
   <td width="220" align="center">
    <img src="https://anosys.ai/assets/img/3.png" width="200">
</td>
    <td valign="middle">
      <h1 style="margin:0;">Investment Research Pipeline - Failure Injection & Root-Cause Analysis (Multi-Agent)</h1>
      <p style="margin:8px 0 0 0;">
        Advanced multi-agent demo: a realistic analyst pipeline with handoffs and a shared workspace, a silently injected context-truncation failure, and a step-by-step root-cause analysis performed entirely from the data collected by AnoSys.
      </p>
    </td>
  </tr>
</table>


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Anosys-AI/anosys-sdk/blob/main/examples/anosys_openai_research_pipeline_demo.ipynb)

This notebook extends the [OpenAI Agents PoC](https://github.com/Anosys-AI/anosys-sdk/blob/main/examples/anosys_openai_agentic_poc.ipynb). It uses the exact same AnoSys pixel (`AnosysOpenAIAgentsLogger` as a trace processor), but builds a **realistic multi-agent application**: an investment-research pipeline in which a research agent, a quant agent, and a memo-writing agent collaborate through handoffs and a shared workspace to produce an investment memo.

The twist: the notebook can **inject a realistic multi-agent failure** — a *silent context truncation* in the shared workspace between agents — behind a single flag. Every agent completes successfully, every handoff works, every tool call returns OK... and the final memo confidently omits the company's most material risks. You will then use the traces collected in AnoSys to find where the information was lost and prove the root cause.

> ⚠️ The company, data and memo in this notebook are entirely fictional and for demonstration purposes only — nothing here is investment advice.

### What you will learn:
1. How to instrument a multi-agent pipeline (handoffs, tools, shared state) with the AnoSys Agents pixel.
2. A production-grade multi-agent pattern: passing large artifacts through a **shared workspace** while stripping tool payloads at handoff boundaries to control context growth.
3. How a **silent truncation bug** in shared state makes downstream agents produce dangerously incomplete output — with all spans green.
4. How to root-cause the failure from AnoSys traces alone: walking the data path across agents, diffing what was *written* against what was *read*.
5. How to verify the fix with a re-run.

## Step 1: Installation

We need the OpenAI Agents SDK and the `anosys-sdk-openai-agents` pixel — the same one used by the basic agentic PoC.

Visit our library for the latest updates and features:
[anosys-sdk-openai-agents on PyPI](https://pypi.org/project/anosys-sdk-openai-agents)

In [ ]:
!pip install -q anosys-sdk-openai-agents openai-agents

## Step 2: Configuration

To run this demo, you'll need two API keys:
1. **OpenAI API Key**: To power the agents.
2. **AnoSys API Key**: To send traces to your observability dashboard.

You can get your AnoSys API key from the [AnoSys Console](https://console.anosys.ai) > Data Ingestion > Integration Options.

In [ ]:
import os

try:
    from google.colab import userdata  # Use Colab secrets to store your keys
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = userdata.get('ANOSYS_API_KEY')
except ImportError:
    # Running outside Colab — fall back to interactive input
    import getpass
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = getpass.getpass("AnoSys API Key: ")

## Step 3: Initialize AnoSys Observability

Same pixel as the agentic PoC: `AnosysOpenAIAgentsLogger` plugs into the Agents SDK as a trace processor, capturing every agent run, LLM generation, tool call and handoff.

As in the single-agent demo, we keep the user context in a mutable dict and **retag `session_id` before each run** (`...-healthy`, `...-failure-injected`, `...-fix-verified`) so the runs are easy to isolate and diff in the AnoSys console.

In [ ]:
from agents.tracing import add_trace_processor
from anosys_sdk_openai_agents import AnosysOpenAIAgentsLogger

# Mutable user context — we update `session_id` before each run so that the
# healthy, failure and fix-verification traces are easy to isolate in AnoSys.
USER_CONTEXT = {
    "session_id": "research-pipeline-setup",
    "token": "demo-user-token",
}

def get_user_context():
    return USER_CONTEXT

processor = AnosysOpenAIAgentsLogger(get_user_context=get_user_context)
add_trace_processor(processor)

print("✅ AnoSys Observability initialized!")

## Step 4: The Dataset — Helix Dynamics (HLX)

The pipeline analyzes **Helix Dynamics**, a (fictional) industrial-robotics company. We give the tools a realistic dataset:

- **Eight quarters of financials** — revenue, cost of goods sold, operating expenses, debt, cash, equity.
- **A recent news feed** — mostly positive items, but containing **two material risks** that any competent memo must flag:
  1. A **patent-infringement lawsuit** seeking $120M in damages.
  2. A **debt-covenant warning** — leverage is close to the limit that would trigger accelerated repayment.

Whether those two items survive the trip through the pipeline is the whole story of this demo.

In [ ]:
FINANCIALS = [  # last 8 quarters, amounts in $M
    {"quarter": "Q3-2024", "revenue": 118.2, "cogs": 71.5, "opex": 31.0,
     "total_debt": 210.0, "cash": 84.0, "shareholders_equity": 74.0},
    {"quarter": "Q4-2024", "revenue": 124.9, "cogs": 74.8, "opex": 32.1,
     "total_debt": 216.0, "cash": 79.5, "shareholders_equity": 76.5},
    {"quarter": "Q1-2025", "revenue": 121.4, "cogs": 73.9, "opex": 33.4,
     "total_debt": 224.0, "cash": 71.2, "shareholders_equity": 75.1},
    {"quarter": "Q2-2025", "revenue": 129.8, "cogs": 78.2, "opex": 34.0,
     "total_debt": 231.0, "cash": 66.8, "shareholders_equity": 77.9},
    {"quarter": "Q3-2025", "revenue": 137.6, "cogs": 82.0, "opex": 35.2,
     "total_debt": 242.0, "cash": 60.1, "shareholders_equity": 79.4},
    {"quarter": "Q4-2025", "revenue": 145.3, "cogs": 86.1, "opex": 36.8,
     "total_debt": 251.0, "cash": 55.7, "shareholders_equity": 81.2},
    {"quarter": "Q1-2026", "revenue": 141.0, "cogs": 85.4, "opex": 38.0,
     "total_debt": 262.0, "cash": 47.9, "shareholders_equity": 80.3},
    {"quarter": "Q2-2026", "revenue": 152.7, "cogs": 89.9, "opex": 39.1,
     "total_debt": 271.0, "cash": 41.5, "shareholders_equity": 82.6},
]

NEWS = [
    {"date": "2026-05-04", "headline": "Helix Dynamics wins $85M automation contract with major EU auto group",
     "body": "Multi-year deployment of Helix's HX-9 robotic arms across four assembly plants, the company's largest contract to date."},
    {"date": "2026-05-21", "headline": "Helix launches HXVision quality-inspection module",
     "body": "New computer-vision add-on targets the growing automated QA market; early pilots report 30% defect-detection improvement."},
    {"date": "2026-06-02", "headline": "Helix Dynamics hires former Fanuc executive as COO",
     "body": "Leadership addition seen as strengthening large-account delivery capability ahead of the EU contract ramp."},
    {"date": "2026-06-17", "headline": "Analyst day: management reiterates 18-20% growth outlook",
     "body": "Management guided FY2026 revenue growth of 18-20%, citing record backlog and the HXVision attach rate."},
    {"date": "2026-06-25", "headline": "VoltEdge Systems sues Helix Dynamics for patent infringement, seeks $120M",
     "body": "Former partner VoltEdge alleges the HX-9 servo-control stack infringes three patents and seeks $120M in damages plus an injunction on HX-9 sales. Helix calls the suit meritless; a preliminary hearing is set for September 2026."},
    {"date": "2026-07-08", "headline": "Q2 filing flags leverage approaching debt-covenant threshold",
     "body": "Helix's 10-Q discloses net leverage of 3.4x against a 3.5x covenant limit on its $150M revolver. Breaching the covenant would allow lenders to accelerate repayment. Management says it is 'evaluating options' including an equity raise."},
]

## Step 5: The Shared Analyst Workspace — and the Failure Injection

In production multi-agent systems, large artifacts (research packets, reports) are rarely passed through the chat context — they are written to a **shared store** (memory layer, scratchpad, vector DB), and agents pass around *references*. We model that with a simple workspace: `workspace_write(key, content)` / `workspace_read(key)`.

**The injected failure** lives in the workspace's persistence layer: a well-intentioned *"context budget guard"* that silently clips any entry larger than `WORKSPACE_CHAR_BUDGET` characters. This is a real production failure class — truncation "optimizations" in memory layers, message buses, or DB columns that drop the tail of a payload without any error.

Note what this failure is **not**:
- It does **not** raise an exception — writes report success.
- It is **size-dependent** — small documents pass through untouched, so the pipeline *mostly works*, which is exactly what makes this class of bug so hard to catch.
- The research packet is structured with **risks in the final section** — so what gets clipped is precisely the material risk disclosure.

In [ ]:
INJECT_FAILURE = False        # ← the only switch. True = enable the buggy context budget guard.
WORKSPACE_CHAR_BUDGET = 1500  # the guard's silent clipping threshold (in characters)

WORKSPACE = {}

def _workspace_store(key: str, content: str) -> str:
    """Persistence layer of the shared analyst workspace.

    The injected failure lives here: a 'context budget guard' that silently
    truncates large entries to WORKSPACE_CHAR_BUDGET characters. The write
    still reports success — the tail of the document is simply gone.
    """
    if INJECT_FAILURE and len(content) > WORKSPACE_CHAR_BUDGET:
        content = content[:WORKSPACE_CHAR_BUDGET]
    WORKSPACE[key] = content
    return content

## Step 6: Tools and Agents

The pipeline has four agents connected by handoffs, mirroring a real analyst team:

1. **PortfolioManagerAgent** (entry point) — routes analysis requests into the pipeline.
2. **ResearchAgent** — pulls financials and news, writes a structured *research packet* to the workspace (risks in section 3), hands off.
3. **QuantAgent** — computes real metrics (growth, margins, leverage) from the raw quarterly data, writes a concise *metrics report* to the workspace, hands off.
4. **MemoAgent** — reads both workspace documents and writes the final memo, which **must** include a Risk Factors section based strictly on the sources.

Two production-grade details worth noticing:
- `compute_metrics` does **real arithmetic** on the dataset — it is not a canned string.
- Handoffs use `handoff_filters.remove_all_tools`, a standard **context-hygiene** pattern: raw tool payloads are stripped at agent boundaries to control context growth, which is precisely why all substantive data must flow through the workspace. (It also means the truncation bug cannot be papered over by leftover context.)

In [ ]:
import json
from agents import Agent, Runner, function_tool, handoff
from agents.extensions import handoff_filters

# --- Tools ---

@function_tool
def get_financials(ticker: str) -> str:
    """Return the last 8 quarters of financial data for a ticker (amounts in $M)."""
    if ticker.upper() != "HLX":
        return json.dumps({"error": f"unknown ticker {ticker}"})
    return json.dumps({"ticker": "HLX", "company": "Helix Dynamics",
                       "quarters": FINANCIALS})

@function_tool
def get_recent_news(ticker: str) -> str:
    """Return recent news items for a ticker, oldest first."""
    if ticker.upper() != "HLX":
        return json.dumps({"error": f"unknown ticker {ticker}"})
    return json.dumps(NEWS)

@function_tool
def compute_metrics(ticker: str) -> str:
    """Compute growth, margin and leverage metrics from the raw quarterly data."""
    if ticker.upper() != "HLX":
        return json.dumps({"error": f"unknown ticker {ticker}"})
    latest, year_ago = FINANCIALS[-1], FINANCIALS[-5]
    ttm = FINANCIALS[-4:]
    ttm_revenue = sum(q["revenue"] for q in ttm)
    ttm_ebit = sum(q["revenue"] - q["cogs"] - q["opex"] for q in ttm)
    metrics = {
        "latest_quarter": latest["quarter"],
        "revenue_yoy_growth_pct": round(
            100 * (latest["revenue"] - year_ago["revenue"]) / year_ago["revenue"], 1),
        "gross_margin_pct": round(
            100 * (latest["revenue"] - latest["cogs"]) / latest["revenue"], 1),
        "operating_margin_pct": round(
            100 * (latest["revenue"] - latest["cogs"] - latest["opex"]) / latest["revenue"], 1),
        "ttm_revenue_musd": round(ttm_revenue, 1),
        "ttm_ebit_musd": round(ttm_ebit, 1),
        "net_debt_musd": round(latest["total_debt"] - latest["cash"], 1),
        "net_leverage_x": round((latest["total_debt"] - latest["cash"]) / ttm_ebit, 2),
        "debt_to_equity_x": round(latest["total_debt"] / latest["shareholders_equity"], 2),
        "cash_musd": latest["cash"],
        "cash_trend_8q": [q["cash"] for q in FINANCIALS],
    }
    return json.dumps(metrics, indent=2)

@function_tool
def workspace_write(key: str, content: str) -> str:
    """Save a document to the shared analyst workspace under `key`."""
    _workspace_store(key, content)
    return json.dumps({"status": "saved", "key": key})

@function_tool
def workspace_read(key: str) -> str:
    """Read a document from the shared analyst workspace."""
    return WORKSPACE.get(key, f"(no document found under key '{key}')")

# --- Agents (defined bottom-up so handoff targets exist) ---

memo_agent = Agent(
    name="MemoAgent",
    model="gpt-4o-mini",
    instructions=(
        "You write the final investment memo. First call workspace_read for "
        "'research_packet' and then for 'metrics_report'. Write the memo STRICTLY "
        "from those two documents — no outside knowledge, no invented facts. "
        "The memo must have exactly these sections: 'Recommendation', "
        "'Financial Analysis', 'Risk Factors'. In Risk Factors, list every risk "
        "mentioned in the source documents; if the sources mention no risks, "
        "write 'No material risks disclosed in the research packet.' "
        "Return the full memo as your final answer."
    ),
    tools=[workspace_read],
)

quant_agent = Agent(
    name="QuantAgent",
    model="gpt-4o-mini",
    instructions=(
        "You are the quantitative analyst. Call workspace_read('research_packet') "
        "for context, then call compute_metrics for the ticker. Write a concise "
        "metrics report (under 1200 characters, plain text) and save it with "
        "workspace_write under the key 'metrics_report'. Then hand off to "
        "MemoAgent. Keep chat replies to one short sentence."
    ),
    tools=[workspace_read, compute_metrics, workspace_write],
    handoffs=[handoff(agent=memo_agent, input_filter=handoff_filters.remove_all_tools)],
)

research_agent = Agent(
    name="ResearchAgent",
    model="gpt-4o-mini",
    instructions=(
        "You are the research analyst. Call get_financials and get_recent_news "
        "for the requested ticker. Then write a research packet with sections in "
        "this exact order: '1. Company Overview', '2. Financial Highlights', "
        "'3. Recent News & Risk Factors'. Section 3 must contain every material "
        "risk you found in the news (lawsuits, covenants, regulatory issues), "
        "each with one sentence of detail. Save the full packet with "
        "workspace_write under the key 'research_packet'. Do NOT repeat the "
        "packet in chat — reply with one short sentence only. Then hand off to "
        "QuantAgent."
    ),
    tools=[get_financials, get_recent_news, workspace_write],
    handoffs=[handoff(agent=quant_agent, input_filter=handoff_filters.remove_all_tools)],
)

pm_agent = Agent(
    name="PortfolioManagerAgent",
    model="gpt-4o-mini",
    instructions=(
        "You coordinate the research pipeline. For any request to analyze a "
        "company or prepare an investment memo, hand off to ResearchAgent "
        "immediately. Do not attempt the analysis yourself."
    ),
    handoffs=[handoff(agent=research_agent)],
)

print("✅ Pipeline assembled: PortfolioManager → Research → Quant → Memo")

## Step 7: Healthy Baseline Run

Run the pipeline with the failure **disabled** to establish the baseline. Expected outcome: a memo whose **Risk Factors** section flags both the VoltEdge lawsuit and the debt-covenant pressure.

**Note:** Since we are in a notebook environment with an existing event loop, we use `await Runner.run()` instead of `Runner.run_sync()`.

In [ ]:
WORKSPACE.clear()
INJECT_FAILURE = False
USER_CONTEXT["session_id"] = "research-pipeline-healthy"

query = "Prepare an investment memo for Helix Dynamics (ticker HLX)."
result = await Runner.run(pm_agent, query, max_turns=25)

print("\n--- FINAL MEMO (healthy run) ---\n")
print(result.final_output)
print(f"\n[demo] research_packet stored length: {len(WORKSPACE.get('research_packet', ''))} chars")

## Step 8: Inject the Failure

Now enable the workspace's buggy *context budget guard* and re-run the **exact same request** from a clean workspace.

What happens under the hood: the ResearchAgent still finds the lawsuit and the covenant warning, still writes a complete packet — but the workspace silently stores only the first 1,500 characters. Section 3 (*Recent News & Risk Factors*) is exactly what falls off the end. The QuantAgent and MemoAgent read the clipped packet, complete "successfully", and the final memo either claims **no material risks** or fills the gap with vague filler.

A confident, professional-looking memo that omits a $120M lawsuit and a covenant breach risk — with every span in the pipeline green. This is the multi-agent failure mode that actually hurts in production.

In [ ]:
WORKSPACE.clear()
INJECT_FAILURE = True               # ☠️ the silent context budget guard is live
USER_CONTEXT["session_id"] = "research-pipeline-failure-injected"

query = "Prepare an investment memo for Helix Dynamics (ticker HLX)."
result = await Runner.run(pm_agent, query, max_turns=25)

print("\n--- FINAL MEMO (failure-injected run) ---\n")
print(result.final_output)
print(f"\n[demo] research_packet stored length: {len(WORKSPACE.get('research_packet', ''))} chars")

## Step 9: Debug & Root-Cause the Failure in AnoSys

The memo lands on a portfolio manager's desk. They happen to know about the VoltEdge suit — and it's not in the memo. Time to investigate in the AnoSys console, using only the collected trace data:

### 1. Isolate the runs
Filter traces by the session IDs we tagged via user context:
- `research-pipeline-healthy`
- `research-pipeline-failure-injected`

### 2. Read the timeline — everything looks fine
The failure run's trace timeline shows the full pipeline: `PortfolioManagerAgent` → handoff → `ResearchAgent` (tools) → handoff → `QuantAgent` (tools) → handoff → `MemoAgent`. Every span has an OK status (`otel_status_code`), normal durations, no `error.type`. Handoffs completed. Nothing failed. **This is why dashboards of error rates alone cannot catch this bug.**

### 3. Start from the symptom and walk the data path backwards
The bad output came from **MemoAgent**, so open its spans first:
- Its `workspace_read("research_packet")` tool span: the **output ends mid-sentence** at ~1,500 characters, and there is **no section '3. Recent News & Risk Factors'** at all.
- Its LLM generation span confirms the model only ever saw the clipped packet — the model faithfully wrote "no material risks" from the evidence it was given. **MemoAgent exonerated.**

### 4. Keep walking upstream to ResearchAgent
- Its `get_recent_news` tool span **output contains both risk items** — the VoltEdge lawsuit and the covenant warning. The data made it into the pipeline. **Source tools exonerated.**
- Its `workspace_write` tool span **input contains the complete packet** — several thousand characters, with section 3 fully written, lawsuit and covenant included. **ResearchAgent exonerated.**

### 5. Corner the bug with a write/read diff
Now the whole failure is pinned between two adjacent spans:

| Evidence | Healthy run | Failure run |
|---|---|---|
| `workspace_write` input (packet length) | ~3–4k chars, has §3 | ~3–4k chars, has §3 |
| `workspace_read` output (packet length) | identical to write | **exactly 1,500 chars, §3 gone** |
| `metrics_report` write vs. read | identical | identical (small doc) |
| MemoAgent prompt tokens (`gen_ai_usage_*`) | baseline | **noticeably lower** |

What was **written** ≠ what was **read**, the clip lands at exactly the budget threshold (a hard cutoff mid-sentence — the signature of truncation, not model behavior), and only the *large* document is affected while the small metrics report survives.

### 6. Root cause
> The workspace persistence layer's "context budget guard" silently truncates entries above 1,500 characters. Because the research packet keeps risks in its final section, the truncation drops precisely the material risk disclosure. Models, prompts, tools, and handoffs are all healthy — the defect is in the shared-state layer between agents.

Fix options (in rising order of robustness): raise the budget, make oversize writes **fail loudly**, or summarize-instead-of-truncate. Never clip silently.

### 7. Production takeaways
- In multi-agent systems, the highest-value traces are at the **boundaries**: handoffs and shared-state reads/writes. AnoSys captures the payloads on both sides, which is what makes the write/read diff possible.
- "All spans green + wrong output" ⇒ walk the **data path**, not the error path.
- Watch payload sizes across boundaries; a downstream drop in prompt tokens for the same task is a cheap, high-signal anomaly.

## Step 10: Fix & Verify

Disable the buggy guard (in real life: ship the fix that fails loudly instead of clipping), retag the session, and re-run the same request. The memo's **Risk Factors** section should again contain the VoltEdge lawsuit and the covenant warning — and the `research-pipeline-fix-verified` traces prove it in the same console view.

In [ ]:
WORKSPACE.clear()
INJECT_FAILURE = False              # ✅ fix deployed
USER_CONTEXT["session_id"] = "research-pipeline-fix-verified"

query = "Prepare an investment memo for Helix Dynamics (ticker HLX)."
result = await Runner.run(pm_agent, query, max_turns=25)

print("\n--- FINAL MEMO (fix verified) ---\n")
print(result.final_output)
print(f"\n[demo] research_packet stored length: {len(WORKSPACE.get('research_packet', ''))} chars")

## Step 11: Explore Your Traces in AnoSys

Head over to your **AnoSys Dashboard**. You should see three groups of traces, one per `session_id`.

### What to look for:
1. **The full pipeline timeline**: agent spans, LLM generations, tool calls and the three handoffs in sequence.
2. **Handoff spans**: control moving `PortfolioManagerAgent` → `ResearchAgent` → `QuantAgent` → `MemoAgent`.
3. **The write/read discrepancy**: `workspace_write` input vs. `workspace_read` output for `research_packet` in the failure run — the entire root cause is visible in these two spans.
4. **Token fingerprints**: MemoAgent's prompt tokens dip in the failure run — the quantitative shadow of the missing context.
5. **User context**: the `session_id` tags that made the run-over-run comparison trivial.

### Troubleshooting
If you don't see traces immediately:
- Ensure your `ANOSYS_API_KEY` is correct. If the key is missing or invalid, you may see `405 Method Not Allowed` errors in the logs.
- Check the output logs in this notebook for any connection errors.
- The SDK sends data asynchronously; it might take a few seconds to appear.